# OCR Output Review Tool
## Step 4: Interactive OCR review
Interactive review of Gemini OCR results with manual correction capabilities.
- Identify low-confidence (< 0.9) and unexpected character text blocks
- View original images for verification
- Correct OCR errors
- Save corrected results

In [ ]:
import json
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
import _02_02_clean_OCR_chars

print("Libraries imported successfully")

In [ ]:
# Setup project paths
script_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
project_root = script_dir if (script_dir / "data").exists() else script_dir.parent

ocr_output_file = project_root / "data" / "02_raw_batch_mass" / "ocr_output_combined_sorted_batch_mass_2026.03.jsonl"
preprocessed_dir = project_root / "data" / "01_preprocessed"
preprocessed_meta_csv = preprocessed_dir / "all_metadata.csv"
review_output_file = project_root / "data" / "02_raw_batch_mass" / "ocr_output_reviewed_batch_mass_2026.03.jsonl"

print(f"Input file exists: {ocr_output_file.exists()}")

In [ ]:
meta_df = pd.read_csv(preprocessed_meta_csv, encoding='utf-8')
print(f"Total metadata records loaded: {len(meta_df)}")
print(meta_df.head())

In [ ]:
# Load OCR output from JSONL, do initial autoclean
ocr_data = []
with open(ocr_output_file, 'r', encoding='utf-8') as f:
    for line in f:
        # print(line)
        if line.strip():
            ocr_data.append(json.loads(line))

df_ocr = pd.DataFrame(ocr_data)

df_ocr["text"] = df_ocr["text"].apply(_02_02_clean_OCR_chars.clean_text)
df_ocr["unexpected_chars"] = df_ocr["text"].apply(_02_02_clean_OCR_chars.has_unexpected_chars)

print(f"Total OCR records loaded: {len(df_ocr)}")
print(df_ocr.info())
print(f"\nFirst few rows:")
print(df_ocr.head())

In [ ]:
# Identify low-confidence items and those with unexpected characters for review
low_conf_threshold = 0.90
df_low_conf = df_ocr[(df_ocr['conf'] < low_conf_threshold) | df_ocr['unexpected_chars']]

print(f"Confidence Score Statistics:")
print(df_ocr['conf'].describe())
print(f"\n---")
print(f"Records with confidence < {low_conf_threshold} or unexpected characters: {len(df_low_conf)} out of {len(df_ocr)}")
print(f"Percentage: {100 * len(df_low_conf) / len(df_ocr):.4f}%")
print(f"\nConfidence distribution:")
print(df_ocr['conf'].value_counts().sort_index(ascending=True))


In [ ]:
df_low_conf.head()

In [ ]:
# Helper function to find original image file
def find_original_image(state, page_num, col_idx):
    """
    Reconstruct the original image path from preprocessed directory.
    Images are stored as: data/01_preprocessed/{state}/{state}_p{page}_r00_c{col}.jpg
    """
    
    # Construct expected filename pattern
    img_path = preprocessed_dir / state /  f"{state}_p{page_num:02d}_r00_c{col_idx:03d}.jpg"
    if img_path.exists():
        return img_path
    
    return None

# Test the function
test_result = find_original_image(df_low_conf.iloc[0]['pub'] if len(df_low_conf) > 0 else 'Texas', 1, 0)
print(f"Test image search result: {test_result}")
print(f"Test path exists: {test_result.exists() if test_result else False}")

In [ ]:
# functions for interactions, a lot of the variables are defined later in the interaction section
def update_display(change=None):
    """Update display when item index changes"""
    idx = current_idx.value
    row = df_low_conf.iloc[idx]
    
    corrected_text.value = row['text']
    confidence_display.value = f"<b>Confidence: {row['conf']:.4f}</b> | Pub: {row['pub']} | Page: {row['page']} | Col: {row['col']}"
    status_display.value = f"<i>Item {idx + 1} of {len(df_low_conf)} | Reviewed: {review_state['reviewed_count']}</i>"

    context_display.value = f"""
    <b>Context:</b><br>
    {df_ocr.loc[df_low_conf.index[idx] - 2: df_low_conf.index[idx] - 1].to_html()}
    <b>{df_ocr.loc[[df_low_conf.index[idx]]].to_html()}</b>
    {df_ocr.loc[df_low_conf.index[idx] + 1: df_low_conf.index[idx] + 2].to_html()}
    """
    
    # Display image
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    if img_path:
        with open(img_path, "rb") as file:
            image_bytes = file.read()
        image_widget.value = image_bytes
def on_save_correction(b):
    idx = current_idx.value
    orig_idx = df_low_conf.index[idx]
    review_state['corrections'][orig_idx] = corrected_text.value.strip()
    review_state['reviewed_count'] += 1
    if idx < len(df_low_conf) - 1:
        current_idx.value = idx + 1
    
def on_skip(b):
    idx = current_idx.value
    review_state['skipped'].add(idx)
    if idx < len(df_low_conf) - 1:
        current_idx.value = idx + 1

def on_previous(b):
    if current_idx.value > 0:
        current_idx.value = current_idx.value - 1

In [ ]:
# render the interactive review interface if there are any low-confidence items
if len(df_low_conf) <= 0:
    print("No low confidence OCR recorded!")
else:
    # Create review state tracking dictionary
    review_state = {
        'current_index': 0,
        'corrections': {},  # Store {original_index: corrected_text}
        'skipped': set(),
        'reviewed_count': 0
    }

    # Create interactive review interface
    current_idx = widgets.IntSlider(value=0, min=0, max=len(df_low_conf)-1, step=1, description='Item:', width='600px')
    corrected_text = widgets.Text(value='', placeholder='Enter corrected text', description='Correction:', width='900px')
    confidence_display = widgets.HTML(value='')
    status_display = widgets.HTML(value='')
    context_display = widgets.HTML(value='')
    character_bank = widgets.HTML(value='▼ ⊕ ◊ ★ † ♁ ‡ △')

    row = df_low_conf.iloc[0]
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    if img_path:
        with open(img_path, "rb") as file:
            image_bytes = file.read()
        image_widget = widgets.Image(
            value=image_bytes,
            format='jpg',
            width=300,
            height=900,
            layout=widgets.Layout(object_fit='scale-down') # Optional styling
        )

    current_idx.observe(update_display, names='value')

    # Buttons for navigation and saving
    btn_save_correction = widgets.Button(description='Save & Next', button_style='success')
    btn_skip = widgets.Button(description='Skip', button_style='info')
    btn_previous = widgets.Button(description='Previous', button_style='warning')

    btn_save_correction.on_click(on_save_correction)
    btn_skip.on_click(on_skip)
    btn_previous.on_click(on_previous)
    buttons_box = widgets.HBox([btn_previous, btn_skip, btn_save_correction])

    control_box = widgets.VBox([
        confidence_display,
        status_display,
        current_idx,
        corrected_text,
        character_bank,
        buttons_box,
        context_display
    ])

    # Display the interface
    update_display()
    display(widgets.HBox([control_box, image_widget]))

In [ ]:
# Summary of corrections
print("="*80)
print("REVIEW SUMMARY")
print("="*80)
print(f"Total low-confidence items: {len(df_low_conf)}")
print(f"Items reviewed & corrected: {len(review_state['corrections'])}")
print(f"Items skipped: {len(review_state['skipped'])}")
print(f"Items not yet reviewed: {len(df_low_conf) - len(review_state['corrections']) - len(review_state['skipped'])}")
# print("\nSample corrections made:")
# for orig_idx, corrected_text in list(review_state['corrections'].items())[:5]:
#     print(f"  Index {orig_idx}: {corrected_text[:60]}")

In [ ]:
# Apply corrections to original data and save to new file
df_reviewed = df_ocr.copy()

# Apply corrections
for orig_idx, corrected_text in review_state['corrections'].items():
    df_reviewed.loc[orig_idx, 'text'] = corrected_text
    df_reviewed.loc[orig_idx, 'conf'] = 1.0  # Set confidence to 1.0 for corrected items

# Mark items as reviewed
#df_reviewed['reviewed'] = df_reviewed.index.isin(review_state['corrections'].keys())

print(f"Total records to save: {len(df_reviewed)}")
print(f"Records with corrections: {df_reviewed['conf'].eq(1.0).sum()}")
print(f"\nSample of reviewed data:")
print(df_reviewed[df_reviewed['conf'] == 1.0].head())

df_reviewed["unexpected_chars"]= df_reviewed["text"].apply(_02_02_clean_OCR_chars.has_unexpected_chars)
unexpected_df = df_reviewed[df_reviewed['unexpected_chars']]
if len(unexpected_df) > 0:
    print(f"*** Remaining text with unexpected characters n=({len(unexpected_df)})!! ***")
else:
    print("No remaining unexpected characters")

In [ ]:
# Save reviewed results to new JSONL file
def save_reviewed_output(df, output_path):
    """Save dataframe to JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for idx, row in df.iterrows():
            # Reconstruct the original entry format
            entry = {
                'pub': row['pub'],
                'page': row['page'],
                'col': row['col'],
                'text': row['text'],
                'conf': row['conf'] if isinstance(row['conf'], (int, float)) else float(row['conf']),
                'x': row['x'],
                'y': row['y'],
                'width': row['width'],
                'height': row['height'],
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    return output_path

# Save the reviewed data
output_path = save_reviewed_output(df_reviewed, review_output_file)
print(f"✓ Reviewed data saved to: {output_path}")
print(f"✓ Total records: {len(df_reviewed)}")
print(f"✓ Corrected records: {df_reviewed['conf'].eq(1.0).sum()}")

# Verify the output file
with open(output_path, 'r', encoding='utf-8') as f:
    sample_lines = [json.loads(line) for line in list(f)[:2]]
    
print(f"\n✓ Verification - First 2 records from new file:")
for i, entry in enumerate(sample_lines):
    print(f"\nRecord {i+1}:")
    print(f"  Text: {entry['text'][:60]}...")
    print(f"  Confidence: {entry['conf']}")
    print(f"  Reviewed: {entry.get('reviewed', False)}")

In [ ]:
# Final statistics and comparison
print("\n" + "="*80)
print("FINAL STATISTICS")
print("="*80)

# Load both files for comparison
df_original = pd.read_json(ocr_output_file, lines=True, encoding='utf-8')
df_final = pd.read_json(review_output_file, lines=True, encoding='utf-8')

print(f"\nOriginal OCR Output:")
print(f"  Total records: {len(df_original)}")
print(f"  Low confidence (<0.90): {(df_original['conf'] < 0.90).sum()}")
print(f"  Have unexpected characters: {df_original["text"].apply(_02_02_clean_OCR_chars.has_unexpected_chars).sum()}")
print(f"  Average confidence: {df_original['conf'].mean():.4f}")

print(f"\nReviewed Output:")
print(f"  Total records: {len(df_final)}")
#print(f"  Reviewed records: {df_final['reviewed'].sum() if 'reviewed' in df_final.columns else 'N/A'}")
print(f"  Low confidence (<0.90): {(df_final['conf'] < 0.90).sum()}")
print(f"  Have unexpected characters: {df_final["text"].apply(_02_02_clean_OCR_chars.has_unexpected_chars).sum()}")
print(f"  Average confidence: {df_final['conf'].mean():.4f}")

print(f"\nOutput saved to: {review_output_file}")
print(f"Ready for next processing step!")